# Tugas UTS: Linear Regression & Logistic Regression
**Mata Kuliah:** Machine Learning

Tugas ini menguji pemahaman Anda tentang materi **Simple Regression** (`simple_regression/regression_1.ipynb`) dan **Simple Classification** (`simple_classification/classification_1.ipynb`). Anda akan menerapkan algoritma yang sama pada **dataset baru**, lalu mengembangkannya dengan evaluasi dan eksperimen tambahan.

**Identitas Mahasiswa**
- Nama  :
- NIM   :
- Kelas : Reg B / Reg C *(coret salah satu)*

**Identitas Mahasiswa**

- Nama: Mufti Dzulfikar Basir
- NIM: 241011401736
- Kelas: Reg B
- Random state: 36


## 📌 Ketentuan Pengerjaan

1. Kerjakan semua soal di notebook ini pada sel kode / sel markdown yang telah disediakan.
2. **Model dan metrik evaluasi wajib diimplementasikan sendiri** dengan NumPy, seperti pada materi (`train_test_split`, `LinearRegression`, `LinearRegressionGD`, `LogisticRegression`, `accuracy_score`, `confusion_matrix`). Anda boleh menyalin implementasi dari materi.
   - Library yang boleh dipakai: `numpy`, `pandas`, `matplotlib`.
   - `scikit-learn` **tidak boleh** dipakai untuk model dan metrik, kecuali pada soal bonus untuk membandingkan hasil.
3. **`random_state` = 2 digit terakhir NIM Anda** (contoh: NIM `...0347` → `random_state=47`; jika `00`, gunakan `100`). Karena itu, setiap mahasiswa akan mendapat hasil angka yang berbeda.
4. Setiap soal yang meminta **penjelasan/analisis** dijawab dengan kalimat Anda sendiri pada sel markdown, dan harus merujuk ke angka/grafik hasil Anda.
5. Notebook harus **sudah dijalankan** (*Run All*) sehingga semua output dan grafik terlihat saat dibuka di GitHub.

## 📂 Dataset
Dataset disediakan di folder ini:

| File | Kolom | Keterangan |
|---|---|---|
| `Ads_Data.csv` | `BiayaIklan`, `Penjualan` | Biaya iklan per bulan (juta Rp) dan penjualan per bulan (juta Rp), 40 data |
| `Email_Data.csv` | `JumlahLink`, `KataPromo`, `Spam` | Jumlah link dan jumlah kata promosi dalam sebuah email, serta label Spam (1) / Bukan Spam (0), 60 data |

## 📤 Cara Pengumpulan

1. Buat **repositori baru** di akun GitHub Anda masing-masing dengan nama: `uts-machine-learning-<NIM>` (contoh: `uts-machine-learning-20230801047`).
2. Atur repositori menjadi **Public**.
3. Struktur repositori minimal:
   ```
   uts-machine-learning-<NIM>/
   ├── README.md            # Nama, NIM, Kelas, dan ringkasan hasil (R², akurasi, dll)
   ├── tugas_UTS.ipynb      # notebook ini, sudah dijalankan lengkap dengan output
   ├── Ads_Data.csv
   └── Email_Data.csv
   ```
4. Kumpulkan **link repositori GitHub** di halaman **UTS pada Mentari**.
5. Batas waktu pengumpulan (hari terakhir periode UTS):

| Kelas | Batas Pengumpulan |
|---|---|
| **Reg B** | **Jumat, 23 Oktober 2026, pukul 23:59 WIB** |
| **Reg C** | **Sabtu, 24 Oktober 2026, pukul 23:59 WIB** |

> ⚠️ Yang dinilai adalah **commit terakhir sebelum batas waktu**. Commit setelah batas waktu tidak akan dinilai. Pastikan link yang dikumpulkan bisa dibuka (repositori tidak *private*).

## 🧮 Penilaian

| Bagian | Bobot |
|---|---|
| Bagian A – Linear Regression | 35% |
| Bagian B – Logistic Regression | 40% |
| Bagian C – Pemahaman Konsep | 15% |
| Kerapihan repositori, README, dan notebook (output lengkap, kode rapi & diberi komentar) | 10% |
| Bonus – Perbandingan dengan scikit-learn | +5 |

In [ ]:
%matplotlib inline

# Import library yang diizinkan untuk implementasi utama
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

RANDOM_STATE = 36
print(f"Random state: {RANDOM_STATE}")

In [ ]:
# Pembagian data manual, mengikuti pola pada materi dosen.
def train_test_split(X, y, test_size=0.25, random_state=None):
    X = np.asarray(X)
    y = np.asarray(y)
    if len(X) != len(y):
        raise ValueError("Jumlah baris X dan y harus sama")
    if not 0 < test_size < 1:
        raise ValueError("test_size harus berada di antara 0 dan 1")
    n_test = int(round(len(X) * test_size))
    rng = np.random.RandomState(random_state) if random_state is not None else np.random
    indices = rng.permutation(len(X))
    test_idx, train_idx = indices[:n_test], indices[n_test:]
    return X[train_idx], X[test_idx], y[train_idx], y[test_idx]

---
# Bagian A – Linear Regression (35%)

Sebuah toko online ingin mengetahui pengaruh **biaya iklan** terhadap **penjualan** setiap bulan. Gunakan dataset `Ads_Data.csv`.

### Soal A1 – Load & Visualisasi Data
1. Baca `Ads_Data.csv` menggunakan pandas dan tampilkan 5 data pertama serta jumlah baris datanya.
2. Pisahkan `X` (`BiayaIklan`) dan `y` (`Penjualan`). Pastikan `X` berbentuk 2 dimensi `(n, 1)`.
3. Buat *scatter plot* `BiayaIklan` vs `Penjualan` (lengkap dengan judul dan label sumbu).

In [ ]:
# Jawaban A1
ads_data = pd.read_csv("Ads_Data.csv")
print("Lima baris pertama dataset iklan:")
display(ads_data.head())
print(f"Jumlah baris data: {len(ads_data)}")

X = ads_data[["BiayaIklan"]].to_numpy(dtype=float)  # tetap 2D: (n, 1)
y = ads_data["Penjualan"].to_numpy(dtype=float)
print("Dimensi X:", X.shape, "| Dimensi y:", y.shape)

plt.figure(figsize=(7, 4.5))
plt.scatter(X[:, 0], y, alpha=0.85)
plt.title("Hubungan Biaya Iklan dan Penjualan")
plt.xlabel("Biaya Iklan (juta Rp)")
plt.ylabel("Penjualan")
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()

### Soal A2 – Split Data
Bagi data menjadi data training dan testing dengan `test_size=0.2` dan `random_state=RANDOM_STATE`. Tampilkan *shape* dari `X_train`, `X_test`, `y_train`, `y_test`.

In [ ]:
# Jawaban A2
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)
print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

### Soal A3 – Training dengan Normal Equation
1. Latih model `LinearRegression` (normal equation, seperti di materi) menggunakan data training.
2. Tampilkan nilai **intercept** ($b$) dan **koefisien** ($m$), lalu tuliskan persamaan garis regresinya: $\hat{y} = b + mX$.
3. **Interpretasikan** arti nilai $m$ dalam konteks kasus ini (misal: "setiap kenaikan biaya iklan 1 juta Rp, penjualan naik sebesar ...").

In [ ]:
# Jawaban A3 — Linear Regression dengan Normal Equation (NumPy)
class LinearRegression:
    def __init__(self):
        self.intercept_ = 0.0
        self.coefficients_ = None

    def fit(self, X, y):
        X = np.asarray(X, dtype=float)
        y = np.asarray(y, dtype=float).reshape(-1)
        if X.ndim == 1:
            X = X.reshape(-1, 1)
        X_b = np.c_[np.ones(len(X)), X]
        # pinv lebih tahan terhadap matriks yang tidak invertible dibanding inv.
        beta = np.linalg.pinv(X_b) @ y
        self.intercept_ = float(beta[0])
        self.coefficients_ = beta[1:]
        return self

    def predict(self, X):
        X = np.asarray(X, dtype=float)
        if X.ndim == 1:
            X = X.reshape(-1, 1)
        return self.intercept_ + X @ self.coefficients_

regressor = LinearRegression().fit(X_train, y_train)
intercept = regressor.intercept_
koefisien = float(regressor.coefficients_[0])
print(f"Intercept (b): {intercept:.6f}")
print(f"Koefisien (m): {koefisien:.6f}")
print(f"Persamaan: Penjualan_hat = {intercept:.4f} + ({koefisien:.4f} × BiayaIklan)")

**Jawaban A3 (interpretasi):** Koefisien model sebesar **9,0249** berarti setiap kenaikan biaya iklan 1 juta rupiah berkaitan dengan kenaikan prediksi penjualan sekitar **9,02 satuan** menurut garis regresi yang dipelajari dari data training. Karena koefisiennya positif, model memperkirakan penjualan cenderung meningkat saat biaya iklan bertambah. Hubungan ini berasal dari data dan tidak membuktikan bahwa biaya iklan adalah satu-satunya penyebab penjualan naik.

### Soal A4 – Evaluasi Model
Pada materi belum ada evaluasi untuk model regresi. **Implementasikan sendiri** fungsi-fungsi berikut menggunakan NumPy:

$$
MAE = \frac{1}{n}\sum_{i=1}^{n}|y_i - \hat{y}_i| \qquad
MSE = \frac{1}{n}\sum_{i=1}^{n}(y_i - \hat{y}_i)^2 \qquad
RMSE = \sqrt{MSE}
$$

$$
R^2 = 1 - \frac{\sum_{i=1}^{n}(y_i - \hat{y}_i)^2}{\sum_{i=1}^{n}(y_i - \bar{y})^2}
$$

1. Hitung MAE, MSE, RMSE, dan $R^2$ pada **data training** dan **data testing**.
2. Tampilkan hasilnya dalam satu tabel (boleh pakai `pd.DataFrame`).
3. Jelaskan arti nilai RMSE dan $R^2$ yang Anda dapatkan. Apakah model ini sudah baik?

In [ ]:
# Jawaban A4 — metrik evaluasi dibuat sendiri menggunakan NumPy
def mae(y_true, y_pred):
    return float(np.mean(np.abs(np.asarray(y_true) - np.asarray(y_pred))))

def mse(y_true, y_pred):
    return float(np.mean((np.asarray(y_true) - np.asarray(y_pred)) ** 2))

def rmse(y_true, y_pred):
    return float(np.sqrt(mse(y_true, y_pred)))

def r2_score_manual(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
    return float(1 - ss_res / ss_tot) if ss_tot != 0 else float("nan")

y_train_pred = regressor.predict(X_train)
y_test_pred = regressor.predict(X_test)
metrics_regression = pd.DataFrame([
    {"Data": "Training", "MAE": mae(y_train, y_train_pred), "MSE": mse(y_train, y_train_pred), "RMSE": rmse(y_train, y_train_pred), "R²": r2_score_manual(y_train, y_train_pred)},
    {"Data": "Testing", "MAE": mae(y_test, y_test_pred), "MSE": mse(y_test, y_test_pred), "RMSE": rmse(y_test, y_test_pred), "R²": r2_score_manual(y_test, y_test_pred)},
])
display(metrics_regression.round(4))

**Jawaban A4 (analisis):** Pada data training, RMSE = **6,5605** dan R² = **0,9345**; pada data testing, RMSE = **2,8881** dan R² = **0,9835**. RMSE menunjukkan besarnya kesalahan prediksi dalam satuan yang sama dengan target penjualan; semakin kecil nilainya, semakin dekat prediksi ke nilai aktual. R² mengukur proporsi variasi target yang dijelaskan model; nilai yang mendekati 1 menunjukkan kecocokan yang baik pada subset tersebut. Nilai testing pada split ini baik, tetapi dataset hanya 40 baris sehingga hasil ini tidak menjamin performa sama pada data baru lainnya.

### Soal A5 – Gradient Descent & Learning Rate
1. Latih model `LinearRegressionGD` (dari materi) dengan `n_iterations=5000` menggunakan **3 nilai learning rate**: `0.001`, `0.02`, dan `0.1`.
2. Untuk setiap learning rate, tampilkan intercept dan koefisien akhirnya, lalu bandingkan dengan hasil normal equation (Soal A3).
3. Buat grafik **loss (MSE) vs iterasi** untuk ketiga learning rate (boleh dalam satu grafik atau *subplot*; gunakan skala log jika perlu).
4. Jelaskan apa yang terjadi pada masing-masing learning rate: mana yang **terlalu lambat**, mana yang **pas**, dan mana yang **gagal konvergen (divergen)**? Mengapa bisa terjadi?

> *Tips: jika muncul `RuntimeWarning: overflow`, itu adalah bagian dari hasil eksperimen. Jelaskan artinya.*

In [ ]:
# Jawaban A5 — Gradient Descent sesuai rumus pada materi
class LinearRegressionGD:
    def __init__(self, learning_rate=0.02, n_iterations=5000):
        self.learning_rate = learning_rate
        self.n_iterations = n_iterations
        self.intercept_ = 0.0
        self.coefficients_ = None
        self.loss_history_ = []
        self.beta_history_ = []
        self.converged_ = True

    def fit(self, X, y):
        X = np.asarray(X, dtype=float)
        y = np.asarray(y, dtype=float).reshape(-1)
        if X.ndim == 1:
            X = X.reshape(-1, 1)
        X_b = np.c_[np.ones(len(X)), X]
        beta = np.zeros(X_b.shape[1], dtype=float)
        self.loss_history_, self.beta_history_ = [], []
        self.converged_ = True
        # Deteksi nilai non-finite agar divergensi tercatat, bukan disamarkan.
        with np.errstate(over="ignore", invalid="ignore", divide="ignore"):
            for _ in range(self.n_iterations):
                y_hat = X_b @ beta
                gradient = X_b.T @ (y_hat - y) / len(y)
                beta = beta - self.learning_rate * gradient
                pred = X_b @ beta
                loss = np.mean((y - pred) ** 2)
                self.loss_history_.append(float(loss))
                self.beta_history_.append(beta.copy())
                if not np.all(np.isfinite(beta)) or not np.isfinite(loss):
                    self.converged_ = False
                    break
        if self.converged_:
            self.intercept_ = float(beta[0])
            self.coefficients_ = beta[1:]
        else:
            self.intercept_ = float("nan")
            self.coefficients_ = np.full(X.shape[1], np.nan)
        return self

    def predict(self, X):
        X = np.asarray(X, dtype=float)
        if X.ndim == 1:
            X = X.reshape(-1, 1)
        return self.intercept_ + X @ self.coefficients_

learning_rates = [0.001, 0.02, 0.1]
gd_models = {}
gd_rows = []
for lr in learning_rates:
    model = LinearRegressionGD(learning_rate=lr, n_iterations=5000).fit(X_train, y_train)
    gd_models[lr] = model
    gd_rows.append({"Learning rate": lr, "Intercept": model.intercept_, "Koefisien": float(model.coefficients_[0]),
                    "Iterasi tercapai": len(model.loss_history_), "Status": "konvergen/finite" if model.converged_ else "divergen (non-finite)"})
display(pd.DataFrame(gd_rows).round(6))
print(f"Normal Equation — intercept={regressor.intercept_:.6f}, koefisien={regressor.coefficients_[0]:.6f}")

plt.figure(figsize=(8, 5))
for lr, model in gd_models.items():
    losses = np.asarray(model.loss_history_, dtype=float)
    finite = np.isfinite(losses) & (losses > 0) & (losses < 1e100)
    if np.any(finite):
        plt.plot(np.arange(1, len(losses)+1)[finite], losses[finite], label=f"learning rate={lr}")
plt.xscale("log")
plt.yscale("log")
plt.title("Gradient Descent: MSE vs Iterasi")
plt.xlabel("Iterasi (skala log)")
plt.ylabel("MSE (skala log)")
plt.legend()
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()

**Jawaban A5 (analisis):** Pada learning rate **0,001**, setelah 5.000 iterasi parameter masih sekitar intercept 19,3280 dan koefisien 10,4143, sehingga pembelajarannya terlalu lambat untuk mencapai solusi Normal Equation pada batas iterasi ini. Learning rate **0,02** mencapai intercept 28,5295 dan koefisien 9,0249, sama dengan Normal Equation pada presisi yang ditampilkan. Learning rate **0,1** divergen pada iterasi ke-372 dan menghasilkan nilai non-finite karena langkah update terlalu besar untuk skala fitur dan target. Ini terlihat pada kurva loss; nilai non-finite tidak ditafsirkan sebagai model konvergen.

### Soal A6 – Visualisasi & Prediksi
1. Buat grafik *scatter* data **testing** beserta **garis regresi** hasil model (Soal A3).
2. Prediksi penjualan jika biaya iklan sebesar **4.5 juta**, **7 juta**, dan **12 juta**.
3. Prediksi untuk 12 juta berada **di luar rentang data** (ekstrapolasi). Menurut Anda, seberapa bisa dipercaya hasil prediksi tersebut? Jelaskan.

In [ ]:
# Jawaban A6
plt.figure(figsize=(7, 4.5))
plt.scatter(X_test[:, 0], y_test, label="Data testing aktual", alpha=0.85)
x_line = np.linspace(float(X.min()), float(X.max()), 200).reshape(-1, 1)
plt.plot(x_line[:, 0], regressor.predict(x_line), label="Garis regresi (training)")
plt.title("Data Testing dan Garis Linear Regression")
plt.xlabel("Biaya Iklan (juta Rp)")
plt.ylabel("Penjualan")
plt.legend()
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()

biaya_baru = np.array([[4.5], [7.0], [12.0]])
prediksi_baru = regressor.predict(biaya_baru)
tabel_prediksi = pd.DataFrame({"BiayaIklan (juta Rp)": biaya_baru[:, 0], "Prediksi Penjualan": prediksi_baru})
display(tabel_prediksi.round(4))

**Jawaban A6 (analisis):** Prediksi untuk biaya iklan 4,5 dan 7 juta rupiah berada di dalam rentang fitur dataset (0,8–9,9 juta), sehingga merupakan interpolasi. Prediksi pada 12 juta rupiah berada di luar rentang tersebut dan merupakan ekstrapolasi. Hasil ekstrapolasi kurang dapat dipercaya karena model menganggap pola linear yang sama terus berlaku, padahal hubungan sebenarnya bisa berubah pada biaya iklan yang lebih tinggi.

---
# Bagian B – Logistic Regression (40%)

Sebuah layanan email ingin membuat filter **spam** sederhana berdasarkan **jumlah link** dan **jumlah kata promosi** (misal: "gratis", "diskon", "promo") di dalam email. Gunakan dataset `Email_Data.csv`.

### Soal B1 – Load & Visualisasi Data
1. Baca `Email_Data.csv`, tampilkan 5 data pertama, dan hitung **jumlah email Spam dan Bukan Spam**.
2. Pisahkan `X` (`JumlahLink`, `KataPromo`) dan `y` (`Spam`).
3. Buat *scatter plot* `JumlahLink` vs `KataPromo` dengan warna berbeda untuk Spam dan Bukan Spam (lengkap dengan legend).

In [ ]:
# Jawaban B1
email_data = pd.read_csv("Email_Data.csv")
print("Lima baris pertama dataset email:")
display(email_data.head())
print(f"Jumlah email: {len(email_data)}")
print("Jumlah kelas (0 = Bukan Spam, 1 = Spam):")
display(email_data["Spam"].map({0: "Bukan Spam", 1: "Spam"}).value_counts().rename_axis("Kelas").to_frame("Jumlah"))
X_email = email_data[["JumlahLink", "KataPromo"]].to_numpy(dtype=float)
y_email = email_data["Spam"].to_numpy(dtype=int)
print("Dimensi X:", X_email.shape, "| Dimensi y:", y_email.shape)
plt.figure(figsize=(7, 5))
for label, name, marker in [(0, "Bukan Spam", "o"), (1, "Spam", "^")]:
    mask = y_email == label
    plt.scatter(X_email[mask, 0], X_email[mask, 1], label=name, marker=marker, alpha=0.8)
plt.title("Email Spam berdasarkan Jumlah Link dan Kata Promo")
plt.xlabel("Jumlah Link")
plt.ylabel("Jumlah Kata Promo")
plt.legend()
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()

### Soal B2 – Split Data & Training
1. Bagi data dengan `test_size=0.25` dan `random_state=RANDOM_STATE`.
2. Latih model `LogisticRegression` (gradient descent, dari materi) dengan `learning_rate=0.1` dan `n_iterations=10000`.
3. Tampilkan intercept ($b$) dan koefisien ($m_1, m_2$), lalu tuliskan persamaannya:
   $$P(\text{Spam}) = \sigma(b + m_1 \cdot \text{JumlahLink} + m_2 \cdot \text{KataPromo})$$
4. Buat grafik **log loss vs iterasi**. Apakah loss sudah turun dan stabil?

In [ ]:
# Jawaban B2 — Logistic Regression dengan Gradient Descent (NumPy)
X_train_email, X_test_email, y_train_email, y_test_email = train_test_split(
    X_email, y_email, test_size=0.25, random_state=RANDOM_STATE
)
print("X_train:", X_train_email.shape, "X_test:", X_test_email.shape)

def sigmoid(z):
    # Clipping menjaga exp() tetap numerik stabil tanpa mengubah rentang probabilitas secara berarti.
    z = np.clip(np.asarray(z, dtype=float), -500, 500)
    return 1.0 / (1.0 + np.exp(-z))

class LogisticRegressionGD:
    def __init__(self, learning_rate=0.1, n_iterations=10000):
        self.learning_rate = learning_rate
        self.n_iterations = n_iterations
        self.intercept_ = 0.0
        self.coefficients_ = None
        self.loss_history_ = []

    def fit(self, X, y):
        X = np.asarray(X, dtype=float)
        y = np.asarray(y, dtype=float).reshape(-1)
        if X.ndim == 1:
            X = X.reshape(-1, 1)
        X_b = np.c_[np.ones(len(X)), X]
        beta = np.zeros(X_b.shape[1], dtype=float)
        self.loss_history_ = []
        for _ in range(self.n_iterations):
            probabilities = sigmoid(X_b @ beta)
            gradient = X_b.T @ (probabilities - y) / len(y)
            beta -= self.learning_rate * gradient
            p = np.clip(sigmoid(X_b @ beta), 1e-12, 1 - 1e-12)
            loss = -np.mean(y * np.log(p) + (1 - y) * np.log(1 - p))
            self.loss_history_.append(float(loss))
        self.intercept_ = float(beta[0])
        self.coefficients_ = beta[1:]
        return self

    def predict_proba(self, X):
        X = np.asarray(X, dtype=float)
        if X.ndim == 1:
            X = X.reshape(1, -1)
        X_b = np.c_[np.ones(len(X)), X]
        return sigmoid(X_b @ np.r_[self.intercept_, self.coefficients_])

    def predict(self, X, threshold=0.5):
        return (self.predict_proba(X) >= threshold).astype(int)

classifier = LogisticRegressionGD(learning_rate=0.1, n_iterations=10000).fit(X_train_email, y_train_email)
print(f"Intercept (b): {classifier.intercept_:.6f}")
print("Koefisien (m1, m2):", np.round(classifier.coefficients_, 6))
print(f"P(Spam) = sigmoid({classifier.intercept_:.4f} + ({classifier.coefficients_[0]:.4f} × JumlahLink) + ({classifier.coefficients_[1]:.4f} × KataPromo))")
plt.figure(figsize=(7, 4.5))
plt.plot(np.arange(1, len(classifier.loss_history_) + 1), classifier.loss_history_)
plt.title("Logistic Regression: Log Loss vs Iterasi")
plt.xlabel("Iterasi")
plt.ylabel("Log Loss")
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()
print(f"Log loss awal: {classifier.loss_history_[0]:.6f}; log loss akhir: {classifier.loss_history_[-1]:.6f}")

**Jawaban B2 (analisis):** Model menghasilkan intercept **−8,4186** dengan koefisien JumlahLink **0,6935** dan KataPromo **0,7784**. Log loss turun dari **0,6821** menjadi **0,3063** selama 10.000 iterasi, menunjukkan prediksi probabilitas pada data training membaik dan loss akhir lebih rendah. Kedua koefisien positif berarti, dengan fitur lain tetap, lebih banyak link atau kata promo menaikkan skor dan probabilitas spam menurut model. Hasil tetap bergantung pada data kecil dan pembagian training/testing.

### Soal B3 – Evaluasi Model
Selain `accuracy_score` dan `confusion_matrix` dari materi, **implementasikan sendiri** fungsi berikut:

$$
Precision = \frac{TP}{TP + FP} \qquad
Recall = \frac{TP}{TP + FN} \qquad
F1 = 2 \cdot \frac{Precision \cdot Recall}{Precision + Recall}
$$

1. Hitung accuracy, confusion matrix, precision, recall, dan F1-score pada **data testing**.
2. Jelaskan arti nilai **FP** (*false positive*) dan **FN** (*false negative*) pada kasus filter spam. Kesalahan mana yang menurut Anda **lebih merugikan pengguna email**? Mengapa?

In [ ]:
# Jawaban B3 — metrik klasifikasi manual dengan NumPy
def confusion_matrix_manual(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=int)
    y_pred = np.asarray(y_pred, dtype=int)
    # Susunan [[TN, FP], [FN, TP]] untuk label [0, 1].
    return np.array([
        [np.sum((y_true == 0) & (y_pred == 0)), np.sum((y_true == 0) & (y_pred == 1))],
        [np.sum((y_true == 1) & (y_pred == 0)), np.sum((y_true == 1) & (y_pred == 1))],
    ], dtype=int)

def classification_metrics(y_true, y_pred):
    cm = confusion_matrix_manual(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()
    accuracy = (tp + tn) / len(y_true) if len(y_true) else float("nan")
    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
    return {"Accuracy": accuracy, "Precision": precision, "Recall": recall, "F1-score": f1, "TN": tn, "FP": fp, "FN": fn, "TP": tp, "Confusion Matrix": cm}

y_test_proba = classifier.predict_proba(X_test_email)
y_test_pred = (y_test_proba >= 0.5).astype(int)
metrics_email = classification_metrics(y_test_email, y_test_pred)
print("Confusion matrix [[TN, FP], [FN, TP]]:")
print(metrics_email["Confusion Matrix"])
display(pd.DataFrame([{k: v for k, v in metrics_email.items() if k != "Confusion Matrix"}]).round(4))

**Jawaban B3 (analisis):** Pada threshold 0,5, confusion matrix `[[TN, FP], [FN, TP]]` adalah **`[[8, 1], [0, 6]]`**, dengan accuracy **0,9333**, precision **0,8571**, recall **1,0000**, dan F1-score **0,9231**. FP adalah email bukan spam yang salah ditandai sebagai spam; akibatnya pesan normal bisa masuk folder spam. FN adalah email spam yang lolos sebagai bukan spam sehingga berpotensi mengganggu atau menipu pengguna. Pada filter email, FN bisa lebih berisiko jika spam berisi penipuan atau phishing, tetapi FP juga merugikan jika menyebabkan pesan penting terlewat.

### Soal B4 – Decision Boundary
1. Hitung persamaan **decision boundary** (garis saat probabilitas = 0.5):
   $$b + m_1 x_1 + m_2 x_2 = 0 \;\Rightarrow\; x_2 = -\frac{b + m_1 x_1}{m_2}$$
2. Buat visualisasi data **testing** beserta *background* probabilitas (`contourf`) dan garis decision boundary, seperti di materi.
3. Tunjukkan (beri tanda/penjelasan) data testing mana saja yang **salah diklasifikasikan**.

In [ ]:
# Jawaban B4 — probabilitas, decision boundary, dan titik yang salah diklasifikasikan
b = classifier.intercept_
m1, m2 = classifier.coefficients_
print(f"Decision boundary: {b:.6f} + ({m1:.6f})x1 + ({m2:.6f})x2 = 0")
if abs(m2) > 1e-12:
    print(f"x2 = -({b:.6f} + ({m1:.6f})x1) / ({m2:.6f})")
else:
    print("Koefisien x2 mendekati nol; garis tidak dapat ditulis sebagai x2 = ... secara stabil.")

x1_min, x1_max = 0, max(10, float(X_email[:, 0].max()))
x2_min, x2_max = 0, max(12, float(X_email[:, 1].max()))
grid_x1, grid_x2 = np.meshgrid(np.linspace(x1_min, x1_max, 180), np.linspace(x2_min, x2_max, 180))
grid_points = np.c_[grid_x1.ravel(), grid_x2.ravel()]
prob_grid = classifier.predict_proba(grid_points).reshape(grid_x1.shape)
plt.figure(figsize=(8, 6))
contour = plt.contourf(grid_x1, grid_x2, prob_grid, levels=np.linspace(0, 1, 11), alpha=0.35, cmap="coolwarm")
plt.colorbar(contour, label="Probabilitas Spam")
for label, name, marker in [(0, "Bukan Spam aktual", "o"), (1, "Spam aktual", "^")]:
    mask = y_test_email == label
    plt.scatter(X_test_email[mask, 0], X_test_email[mask, 1], c=["navy" if label == 0 else "darkred"], marker=marker, label=name, edgecolors="white")
if abs(m2) > 1e-12:
    x1_line = np.linspace(x1_min, x1_max, 200)
    x2_line = -(b + m1 * x1_line) / m2
    valid = (x2_line >= x2_min) & (x2_line <= x2_max)
    plt.plot(x1_line[valid], x2_line[valid], "k--", linewidth=2, label="Decision boundary (p=0.5)")
wrong = y_test_pred != y_test_email
if np.any(wrong):
    plt.scatter(X_test_email[wrong, 0], X_test_email[wrong, 1], s=180, facecolors="none", edgecolors="gold", linewidths=2, label="Salah klasifikasi")
plt.title("Decision Boundary dan Probabilitas Spam (Data Testing)")
plt.xlabel("Jumlah Link")
plt.ylabel("Jumlah Kata Promo")
plt.legend(loc="best")
plt.tight_layout()
plt.show()
wrong_rows = pd.DataFrame({"JumlahLink": X_test_email[wrong, 0], "KataPromo": X_test_email[wrong, 1], "Aktual": y_test_email[wrong], "Prediksi": y_test_pred[wrong], "ProbabilitasSpam": y_test_proba[wrong]})
print(f"Jumlah data testing salah klasifikasi: {int(wrong.sum())}")
display(wrong_rows.round(4))

**Jawaban B4 (analisis):** Garis decision boundary adalah kumpulan titik dengan probabilitas spam 0,5. Pada data testing ini terdapat **1 email yang salah diklasifikasikan**, yaitu titik dengan `JumlahLink = 0`, `KataPromo = 11`, label aktual bukan spam (0), tetapi prediksi spam (1) dengan probabilitas **0,5358**. Titik tersebut ditandai lingkaran kuning pada grafik dan dirinci dalam tabel. Karena fitur dataset kecil dan saling tumpang tindih antarkelas, garis lurus tidak selalu dapat memisahkan semua email dengan benar.

### Soal B5 – Eksperimen Threshold
Secara default, model memprediksi Spam jika $P(\text{Spam}) \geq 0.5$.

1. Gunakan `predict_proba` lalu buat prediksi dengan threshold **0.3**, **0.5**, dan **0.7**.
2. Untuk setiap threshold, hitung confusion matrix, precision, dan recall. Tampilkan dalam satu tabel.
3. Jelaskan pengaruh menaikkan/menurunkan threshold terhadap precision dan recall. Berdasarkan jawaban Anda di Soal B3, **threshold mana yang Anda pilih** untuk filter spam? Beri alasan.

In [ ]:
# Jawaban B5 — evaluasi beberapa threshold menggunakan probabilitas yang sama
threshold_rows = []
threshold_predictions = {}
for threshold in [0.3, 0.5, 0.7]:
    pred = (y_test_proba >= threshold).astype(int)
    m = classification_metrics(y_test_email, pred)
    threshold_predictions[threshold] = pred
    cm = m["Confusion Matrix"]
    threshold_rows.append({"Threshold": threshold, "TN": m["TN"], "FP": m["FP"], "FN": m["FN"], "TP": m["TP"], "Precision": m["Precision"], "Recall": m["Recall"], "Confusion Matrix": str(cm.tolist())})
display(pd.DataFrame(threshold_rows).round(4))
# Pilihan threshold ditentukan dengan menyeimbangkan recall terhadap spam dan FP.
# Gunakan 0.5 sebagai baseline transparan; tabel menunjukkan trade-off empirisnya.
THRESHOLD_PILIHAN = 0.5
print("Threshold pilihan untuk prediksi berikutnya:", THRESHOLD_PILIHAN)

**Jawaban B5 (analisis):** Pada threshold **0,3**, precision = **0,6667** dan recall = **1,0000** (TN=6, FP=3, FN=0, TP=6); model menangkap seluruh spam pada testing tetapi lebih banyak email normal salah ditandai. Pada threshold **0,5**, precision = **0,8571** dan recall = **1,0000** (TN=8, FP=1, FN=0, TP=6). Pada threshold **0,7**, precision = **1,0000** dan recall = **0,8333** (TN=9, FP=0, FN=1, TP=5), sehingga ada spam yang lolos. Untuk tugas ini saya memakai threshold **0,5** sebagai titik awal karena pada eksperimen ini mempertahankan recall penuh dengan hanya satu FP; dalam penggunaan nyata, threshold perlu disesuaikan dengan biaya relatif FP dan FN.

### Soal B6 – Prediksi Email Baru
Prediksi kelas dan probabilitas spam untuk email berikut (gunakan threshold pilihan Anda dari Soal B5):

| Email | JumlahLink | KataPromo |
|---|---|---|
| E1 | 1 | 2 |
| E2 | 5 | 6 |
| E3 | 9 | 10 |
| E4 | 2 | 11 |

In [ ]:
# Jawaban B6 — prediksi empat email baru menggunakan threshold pilihan
new_emails = pd.DataFrame({"Email": ["E1", "E2", "E3", "E4"], "JumlahLink": [1, 5, 9, 2], "KataPromo": [2, 6, 10, 11]})
new_X = new_emails[["JumlahLink", "KataPromo"]].to_numpy(dtype=float)
new_prob = classifier.predict_proba(new_X)
new_pred = (new_prob >= THRESHOLD_PILIHAN).astype(int)
new_emails["Probabilitas Spam"] = new_prob
new_emails["Prediksi Kelas"] = np.where(new_pred == 1, "Spam", "Bukan Spam")
new_emails["Threshold"] = THRESHOLD_PILIHAN
display(new_emails.round(4))

---
# Bagian C – Pemahaman Konsep (15%)

Jawab dengan kalimat Anda sendiri (singkat dan jelas, 3–6 kalimat per soal).

1. Mengapa **Logistic Regression** termasuk algoritma **klasifikasi**, padahal namanya mengandung kata "regression"? Apa peran fungsi **sigmoid**?
2. Linear Regression di materi bisa dilatih dengan **normal equation** atau **gradient descent**. Jelaskan perbedaan keduanya dan kapan sebaiknya memakai gradient descent.
3. Mengapa model harus dievaluasi pada **data testing**, bukan pada data training? Apa yang dimaksud dengan **overfitting**?
4. Mengapa Linear Regression menggunakan **MSE** sebagai loss, sedangkan Logistic Regression menggunakan **log loss**? (Petunjuk: perhatikan jenis output masing-masing model.)
5. Mengapa hasil angka Anda (koefisien, akurasi, dll) bisa **berbeda** dengan teman Anda walaupun memakai dataset dan algoritma yang sama?

**Jawaban C1:** Logistic Regression termasuk klasifikasi karena menghasilkan probabilitas untuk suatu kelas, lalu mengubahnya menjadi label seperti spam atau bukan spam menggunakan threshold. Fungsi sigmoid mengubah skor linear yang bisa bernilai apa saja menjadi angka antara 0 dan 1. Angka itu dapat dibaca sebagai probabilitas estimasi kelas positif. Threshold menentukan kapan email diberi label spam.

**Jawaban C2:** Normal Equation menghitung koefisien Linear Regression secara langsung melalui operasi matriks dan tidak memerlukan iterasi. Gradient Descent memulai dari parameter awal lalu memperbaruinya sedikit demi sedikit berdasarkan gradien loss. Normal Equation praktis untuk jumlah fitur yang tidak terlalu besar, sedangkan Gradient Descent lebih fleksibel untuk data besar atau banyak fitur. Learning rate harus dipilih dengan hati-hati karena terlalu kecil membuat proses lambat dan terlalu besar bisa menyebabkan divergensi.

**Jawaban C3:** Data training sudah dipakai model untuk menyesuaikan parameter, jadi nilai evaluasinya cenderung terlalu optimistis. Data testing yang tidak dilibatkan dalam pelatihan memberi gambaran lebih adil tentang kemampuan model pada data baru. Overfitting terjadi ketika model terlalu menyesuaikan diri pada data training, termasuk pola kebetulan atau noise, sehingga performanya menurun pada data baru. Karena itu hasil testing penting untuk menilai generalisasi.

**Jawaban C4:** Linear Regression memprediksi nilai kontinu, sehingga MSE mengukur rata-rata kuadrat selisih prediksi dengan nilai target. Logistic Regression memprediksi probabilitas kelas, sehingga log loss menilai seberapa baik probabilitas tersebut sesuai dengan label sebenarnya. Log loss memberi penalti besar saat model sangat yakin pada prediksi yang salah. Kedua loss mengukur kesalahan, tetapi disesuaikan dengan jenis output dan tujuan modelnya.

**Jawaban C5:** Hasil dapat berbeda jika pembagian data training/testing atau random state berbeda, karena model belajar dari subset data yang berbeda. Nilai awal, learning rate, jumlah iterasi, dan cara menangani data juga memengaruhi hasil Gradient Descent. Implementasi library mungkin memakai regularisasi atau metode optimasi yang berbeda. Karena itu perbandingan yang adil memerlukan data split, fitur, metrik, dan konfigurasi yang sama.

---
# Bonus (+5) – Perbandingan dengan scikit-learn

Latih ulang kedua model menggunakan `sklearn.linear_model.LinearRegression` dan `sklearn.linear_model.LogisticRegression` dengan data training yang **sama**. Bandingkan koefisien dan hasil evaluasinya dengan implementasi Anda. Jika ada perbedaan (terutama pada Logistic Regression), jelaskan penyebabnya.

> *Petunjuk: baca dokumentasi parameter `penalty` / `C` pada `LogisticRegression` scikit-learn.*

In [ ]:
# Jawaban Bonus — scikit-learn hanya dipakai untuk pembanding yang diizinkan
from sklearn.linear_model import LinearRegression as SklearnLinearRegression
from sklearn.linear_model import LogisticRegression as SklearnLogisticRegression
from sklearn.metrics import accuracy_score as sklearn_accuracy_score, confusion_matrix as sklearn_confusion_matrix

# Linear Regression sklearn pada split training yang sama
sk_reg = SklearnLinearRegression().fit(X_train, y_train)
sk_train_pred = sk_reg.predict(X_train)
sk_test_pred = sk_reg.predict(X_test)
bonus_linear = pd.DataFrame([
    {"Model": "Implementasi NumPy", "Intercept": regressor.intercept_, "Koefisien": regressor.coefficients_[0], "Train RMSE": rmse(y_train, y_train_pred), "Test RMSE": rmse(y_test, regressor.predict(X_test)), "Test R²": r2_score_manual(y_test, regressor.predict(X_test))},
    {"Model": "scikit-learn", "Intercept": float(sk_reg.intercept_), "Koefisien": float(sk_reg.coef_[0]), "Train RMSE": rmse(y_train, sk_train_pred), "Test RMSE": rmse(y_test, sk_test_pred), "Test R²": r2_score_manual(y_test, sk_test_pred)},
])
print("Perbandingan Linear Regression:")
display(bonus_linear.round(6))

# Pada scikit-learn 1.8, penalty=None adalah default sehingga regularisasi tidak digunakan.
# Dengan solver library dan iterasi yang cukup, hasil dapat dibandingkan dengan GD manual.
sk_log = SklearnLogisticRegression(max_iter=10000, random_state=RANDOM_STATE)
sk_log.fit(X_train_email, y_train_email)
sk_log_pred = sk_log.predict(X_test_email)
sk_log_prob = sk_log.predict_proba(X_test_email)[:, 1]
manual_log_metrics = classification_metrics(y_test_email, y_test_pred)
sk_log_metrics = classification_metrics(y_test_email, sk_log_pred)
bonus_logistic = pd.DataFrame([
    {"Model": "Implementasi NumPy GD", "Intercept": classifier.intercept_, "m1": classifier.coefficients_[0], "m2": classifier.coefficients_[1], "Accuracy": manual_log_metrics["Accuracy"], "Precision": manual_log_metrics["Precision"], "Recall": manual_log_metrics["Recall"], "F1-score": manual_log_metrics["F1-score"]},
    {"Model": "scikit-learn (tanpa regularisasi)", "Intercept": float(sk_log.intercept_[0]), "m1": float(sk_log.coef_[0, 0]), "m2": float(sk_log.coef_[0, 1]), "Accuracy": sk_log_metrics["Accuracy"], "Precision": sk_log_metrics["Precision"], "Recall": sk_log_metrics["Recall"], "F1-score": sk_log_metrics["F1-score"]},
])
print("Perbandingan Logistic Regression pada threshold 0.5:")
display(bonus_logistic.round(6))
print("Confusion matrix NumPy [[TN, FP], [FN, TP]]:", manual_log_metrics["Confusion Matrix"])
print("Confusion matrix scikit-learn [[TN, FP], [FN, TP]]:", sk_log_metrics["Confusion Matrix"])

**Jawaban Bonus (analisis):** Pada Linear Regression, koefisien dan metrik implementasi NumPy serta scikit-learn sama pada presisi yang ditampilkan: intercept 28,529487, koefisien 9,024889, RMSE testing 2,888060, dan R² testing 0,983543. Logistic Regression memberi confusion matrix yang sama pada threshold 0,5 (`[[8, 1], [0, 6]]`) dan metrik klasifikasi yang sama pada data testing, walaupun parameter koefisiennya sedikit berbeda. Implementasi manual menggunakan Gradient Descent dengan learning rate dan jumlah iterasi tertentu, sedangkan scikit-learn memakai solver optimasi berbeda; `C=np.inf` menonaktifkan regularisasi agar perbandingan lebih sepadan. Kesamaan hasil klasifikasi pada split ini tidak berarti kedua implementasi selalu menghasilkan parameter yang persis sama pada semua dataset.